# Optimization Note 5: Nonlinear Constraints

**Goal:** Extend the primal-dual IPM to the general nonlinear programming (NLP) problem with nonlinear equality and inequality constraints.

## The General NLP

$$\min_{x \in \mathbb{R}^n} f(x)$$
$$\text{s.t.} \quad g_L \leq g(x) \leq g_U$$
$$\phantom{\text{s.t.}} \quad x_L \leq x \leq x_U$$

This is the problem formulation that IPOPT and ripopt solve. It covers:
- **Equalities:** set $g_{L,j} = g_{U,j}$
- **One-sided inequalities:** set one bound to $\pm \infty$
- **Two-sided inequalities:** both bounds finite
- **Free variables:** set $x_L = -\infty, x_U = +\infty$

## Handling Inequalities: Slack Variables

Convert inequality $g_j(x) \leq g_{U,j}$ to equality by introducing a slack $s_j \geq 0$:

$$g_j(x) + s_j = g_{U,j}, \quad s_j \geq 0$$

Now the barrier applies to the slack: $-\mu \ln(s_j)$.

The beauty of this approach: **every NLP reduces to equality constraints plus bounds**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6, suppress=True)

## The Full KKT System

For the general NLP with bounds on both $x$ and constraint slacks, the KKT system becomes:

$$\begin{bmatrix} H + \Sigma_x & J^T \\ J & -\Sigma_s^{-1} \end{bmatrix} \begin{bmatrix} \Delta x \\ \Delta y \end{bmatrix} = -\begin{bmatrix} r_d \\ r_p \end{bmatrix}$$

where:
- $H = \nabla^2_{xx} \mathcal{L}$ = Hessian of Lagrangian
- $\Sigma_x$ = barrier diagonal for variable bounds
- $\Sigma_s^{-1}$ = barrier diagonal for constraint slacks (makes $(2,2)$ block negative!)
- $J$ = constraint Jacobian
- $r_d$ = dual residual (stationarity violation)
- $r_p$ = primal residual (constraint violation)

### The Lagrangian

$$\mathcal{L}(x, y) = f(x) + \sum_j y_j \, g_j(x)$$

Its Hessian includes **second-order constraint information**:

$$H = \nabla^2 f(x) + \sum_j y_j \nabla^2 g_j(x)$$

This is why we need the Hessian of each constraint, not just the Jacobian.

In [ ]:
class NLPProblem:
    """A simple NLP problem definition."""
    
    def __init__(self, n, m):
        self.n = n  # variables
        self.m = m  # constraints
        self.x_lower = np.full(n, -np.inf)
        self.x_upper = np.full(n, np.inf)
        self.g_lower = np.full(m, -np.inf)
        self.g_upper = np.full(m, np.inf)


def solve_nlp(problem, f, grad_f, hess_lag, g, jac_g, x0,
              mu_init=0.1, tol=1e-8, max_iter=200):
    """Primal-dual IPM for general NLP.
    
    Handles: equalities, one-sided and two-sided inequalities, variable bounds.
    """
    n = problem.n
    m = problem.m
    xl, xu = problem.x_lower, problem.x_upper
    gl, gu = problem.g_lower, problem.g_upper
    
    x = np.array(x0, dtype=float)
    
    # Push x inside bounds
    for i in range(n):
        if np.isfinite(xl[i]): x[i] = max(x[i], xl[i] + 0.01)
        if np.isfinite(xu[i]): x[i] = min(x[i], xu[i] - 0.01)
    
    # Initialize multipliers
    y = np.zeros(m)
    
    # Bound multipliers for x
    has_xl = np.isfinite(xl)
    has_xu = np.isfinite(xu)
    zl = np.where(has_xl, 0.1, 0.0)  # lower bound multipliers
    zu = np.where(has_xu, 0.1, 0.0)  # upper bound multipliers
    
    mu = mu_init
    history = []
    
    for k in range(max_iter):
        # Evaluate
        gval = g(x)
        gf = grad_f(x)
        J = jac_g(x)
        H = hess_lag(x, y)
        
        # Slacks
        sl = x - xl  # slack to lower bound
        su = xu - x  # slack to upper bound
        
        # Barrier diagonal for variable bounds
        sigma = np.zeros(n)
        for i in range(n):
            if has_xl[i]: sigma[i] += zl[i] / sl[i]
            if has_xu[i]: sigma[i] += zu[i] / su[i]
        
        # Constraint slack barrier diagonal (for inequality constraints)
        sigma_s = np.zeros(m)
        for j in range(m):
            is_eq = (gl[j] == gu[j])
            if not is_eq:
                # Inequality: compute slack-based barrier
                if np.isfinite(gl[j]):
                    s_low = max(gval[j] - gl[j], 1e-8)
                    sigma_s[j] += mu / s_low**2
                if np.isfinite(gu[j]):
                    s_up = max(gu[j] - gval[j], 1e-8)
                    sigma_s[j] += mu / s_up**2
        
        # Residuals
        r_dual = gf + J.T @ y - zl + zu
        r_primal = np.zeros(m)
        for j in range(m):
            is_eq = (gl[j] == gu[j])
            if is_eq:
                r_primal[j] = gval[j] - gl[j]
            else:
                # Target: move toward feasibility
                if gval[j] < gl[j]:
                    r_primal[j] = gval[j] - gl[j]
                elif gval[j] > gu[j]:
                    r_primal[j] = gval[j] - gu[j]
                else:
                    r_primal[j] = 0.0  # feasible
        
        # Convergence
        dual_inf = np.linalg.norm(r_dual, np.inf)
        primal_inf = np.linalg.norm(r_primal, np.inf)
        compl = 0.0
        for i in range(n):
            if has_xl[i]: compl = max(compl, abs(zl[i] * sl[i] - mu))
            if has_xu[i]: compl = max(compl, abs(zu[i] * su[i] - mu))
        
        history.append({
            'x': x.copy(), 'f': f(x), 'mu': mu,
            'dual_inf': dual_inf, 'primal_inf': primal_inf, 'compl': compl
        })
        
        if dual_inf < tol and primal_inf < tol and compl < tol:
            break
        
        # Build and solve KKT system
        KKT = np.zeros((n + m, n + m))
        KKT[:n, :n] = H + np.diag(sigma)
        KKT[:n, n:] = J.T
        KKT[n:, :n] = J
        for j in range(m):
            if sigma_s[j] > 0:
                KKT[n+j, n+j] = -1.0 / sigma_s[j]
        
        # RHS
        rhs_d = -gf - J.T @ y
        for i in range(n):
            if has_xl[i]: rhs_d[i] += zl[i] - mu / sl[i]
            if has_xu[i]: rhs_d[i] -= zu[i] - mu / su[i]
        rhs_p = -r_primal
        
        rhs = np.concatenate([rhs_d, rhs_p])
        
        try:
            sol = np.linalg.solve(KKT, rhs)
        except np.linalg.LinAlgError:
            KKT[:n, :n] += 1e-6 * np.eye(n)  # regularize
            sol = np.linalg.solve(KKT, rhs)
        
        dx = sol[:n]
        dy = sol[n:]
        
        # Compute dz from complementarity
        dzl = np.zeros(n)
        dzu = np.zeros(n)
        for i in range(n):
            if has_xl[i]: dzl[i] = (mu - zl[i] * sl[i] - zl[i] * dx[i]) / sl[i]
            if has_xu[i]: dzu[i] = (mu - zu[i] * su[i] + zu[i] * dx[i]) / su[i]
        
        # Fraction-to-boundary
        tau = max(0.99, 1 - mu)
        alpha_p = 1.0
        for i in range(n):
            if has_xl[i] and dx[i] < 0:
                alpha_p = min(alpha_p, -tau * sl[i] / dx[i])
            if has_xu[i] and dx[i] > 0:
                alpha_p = min(alpha_p, tau * su[i] / dx[i])
        
        alpha_d = 1.0
        for i in range(n):
            if has_xl[i] and dzl[i] < 0:
                alpha_d = min(alpha_d, -tau * zl[i] / dzl[i])
            if has_xu[i] and dzu[i] < 0:
                alpha_d = min(alpha_d, -tau * zu[i] / dzu[i])
        
        # Update
        x += alpha_p * dx
        y += alpha_d * dy
        zl += alpha_d * dzl
        zu += alpha_d * dzu
        
        # Update mu
        avg_compl = 0.0
        count = 0
        for i in range(n):
            if has_xl[i]: avg_compl += zl[i] * (x[i] - xl[i]); count += 1
            if has_xu[i]: avg_compl += zu[i] * (xu[i] - x[i]); count += 1
        if count > 0:
            avg_compl /= count
            mu = min(0.2 * mu, avg_compl / 10.0)
        else:
            mu *= 0.2
        mu = max(mu, 1e-12)
    
    return x, y, history

## Example: HS071 — The Classic IPOPT Test Problem

This is the canonical test problem for interior point solvers:

$$\min \; x_1 x_4 (x_1 + x_2 + x_3) + x_3$$
$$\text{s.t.} \quad x_1 x_2 x_3 x_4 \geq 25$$
$$\phantom{\text{s.t.}} \quad x_1^2 + x_2^2 + x_3^2 + x_4^2 = 40$$
$$\phantom{\text{s.t.}} \quad 1 \leq x_i \leq 5$$

In [ ]:
# HS071
prob = NLPProblem(n=4, m=2)
prob.x_lower = np.array([1.0, 1.0, 1.0, 1.0])
prob.x_upper = np.array([5.0, 5.0, 5.0, 5.0])
prob.g_lower = np.array([25.0, 40.0])
prob.g_upper = np.array([np.inf, 40.0])  # g1 >= 25, g2 = 40

def f71(x):
    return x[0] * x[3] * (x[0] + x[1] + x[2]) + x[2]

def grad_f71(x):
    return np.array([
        x[3] * (2*x[0] + x[1] + x[2]),
        x[0] * x[3],
        x[0] * x[3] + 1,
        x[0] * (x[0] + x[1] + x[2])
    ])

def g71(x):
    return np.array([
        x[0] * x[1] * x[2] * x[3],  # >= 25
        x[0]**2 + x[1]**2 + x[2]**2 + x[3]**2  # = 40
    ])

def jac_g71(x):
    return np.array([
        [x[1]*x[2]*x[3], x[0]*x[2]*x[3], x[0]*x[1]*x[3], x[0]*x[1]*x[2]],
        [2*x[0], 2*x[1], 2*x[2], 2*x[3]]
    ])

def hess_lag71(x, y):
    H = np.zeros((4, 4))
    # Objective Hessian
    H[0, 0] = 2 * x[3]
    H[0, 1] = x[3]; H[1, 0] = x[3]
    H[0, 2] = x[3]; H[2, 0] = x[3]
    H[0, 3] = 2*x[0] + x[1] + x[2]; H[3, 0] = H[0, 3]
    H[1, 3] = x[0]; H[3, 1] = x[0]
    H[2, 3] = x[0]; H[3, 2] = x[0]
    
    # Constraint 1 Hessian (x1*x2*x3*x4) * y[0]
    if len(y) > 0:
        H[0, 1] += y[0] * x[2]*x[3]; H[1, 0] = H[0, 1]
        H[0, 2] += y[0] * x[1]*x[3]; H[2, 0] = H[0, 2]
        H[0, 3] += y[0] * x[1]*x[2]; H[3, 0] = H[0, 3]
        H[1, 2] += y[0] * x[0]*x[3]; H[2, 1] = H[1, 2]
        H[1, 3] += y[0] * x[0]*x[2]; H[3, 1] = H[1, 3]
        H[2, 3] += y[0] * x[0]*x[1]; H[3, 2] = H[2, 3]
    
    # Constraint 2 Hessian (sum of squares) * y[1]
    if len(y) > 1:
        for i in range(4):
            H[i, i] += 2 * y[1]
    
    return H


x0 = np.array([1.0, 5.0, 5.0, 1.0])
x_opt, y_opt, hist = solve_nlp(prob, f71, grad_f71, hess_lag71, g71, jac_g71, x0)

print(f"Solution: x = {x_opt}")
print(f"f(x*) = {f71(x_opt):.6f}")
print(f"g(x*) = {g71(x_opt)}")
print(f"y = {y_opt}")
print(f"\nExpected: x ≈ [1.0, 4.743, 3.821, 1.379]")
print(f"Expected: f ≈ 17.014")

In [ ]:
# Convergence history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

iters = range(len(hist))

axes[0].semilogy(iters, [h['dual_inf'] for h in hist], 'b.-')
axes[0].set_title('Dual infeasibility')
axes[0].set_xlabel('Iteration')
axes[0].axhline(y=tol if 'tol' in dir() else 1e-8, color='r', linestyle='--', label='tol')

axes[1].semilogy(iters, [max(h['primal_inf'], 1e-16) for h in hist], 'r.-')
axes[1].set_title('Primal infeasibility')
axes[1].set_xlabel('Iteration')

axes[2].semilogy(iters, [h['mu'] for h in hist], 'g.-')
axes[2].set_title('Barrier parameter $\\mu$')
axes[2].set_xlabel('Iteration')

plt.tight_layout()
plt.show()

## The Filter Line Search

With constraints, we have two competing objectives:
1. Decrease $f(x)$ (optimality)
2. Decrease $\|c(x)\|$ (feasibility)

A simple merit function combines them, but choosing the penalty weight is difficult.

The **filter method** (used by IPOPT and ripopt) avoids this by maintaining a set of "acceptable" $(\theta, \varphi)$ pairs:
- $\theta = \|c(x)\|$ (constraint violation)
- $\varphi = f(x) + \text{barrier terms}$ (barrier objective)

A trial point is accepted if it **dominates** at least one entry in the filter — either less constraint violation OR better objective. This is like Pareto optimality in multi-objective optimization.

ripopt's filter combines this with an **Armijo-type switching condition**: if the constraint violation is already small, require sufficient decrease in $\varphi$ instead.

## The Full Problem Interface

Here's what a solver needs from the user for a general NLP. This is essentially the IPOPT/ripopt problem interface:

| Callback | Returns | Cost |
|---|---|---|
| `objective(x)` | $f(x)$ | $O(n)$ typically |
| `gradient(x)` | $\nabla f(x)$ | $O(n)$ |
| `constraints(x)` | $g(x)$ | $O(m)$ |
| `jacobian(x)` | $J(x) = \nabla g(x)^T$ | $O(\text{nnz}(J))$ |
| `hessian(x, y)` | $\nabla^2_{xx} \mathcal{L}(x,y)$ | $O(\text{nnz}(H))$ |
| `bounds` | $x_L, x_U, g_L, g_U$ | Once |

The Jacobian and Hessian are typically **sparse** — and their sparsity pattern is fixed across iterations. This is why the symbolic analysis from the linear solver series (Note 5) is so valuable: analyze the pattern once, refactor cheaply at every iteration.

## What We've Learned

1. **Slack variables** convert inequalities to equalities + bounds: all NLPs reduce to this form
2. The **Lagrangian Hessian** $H = \nabla^2 f + \sum y_j \nabla^2 g_j$ includes second-order constraint information
3. The **KKT matrix** has a $(2,2)$ block from constraint slack barriers, making it indefinite
4. The **filter line search** handles the two competing objectives (optimality vs feasibility) without penalty parameters
5. The problem interface is: objective, gradient, constraints, Jacobian, Hessian, bounds

## What's Next

Our IPM works on well-behaved problems. But real problems are messy: the Hessian may be indefinite, the KKT system may be singular, the starting point may be deeply infeasible. In Note 6, we cover the **robustness machinery** that makes the difference between a textbook algorithm and a production solver: inertia correction, regularization, and restoration.

---

*This is Optimization Note 5 in a series building from Newton's method to the interior point optimizer [ripopt](https://github.com/jkitchin/ripopt).*